# TP · etudiants · Google Colab

Cybersup · Machine Learning Avancé · **Chrys Fé-Marty NIONGOLO** · septembre 2026

Ce notebook est autonome : aucun clonage, jeton GitHub ou fichier du dépôt n'est nécessaire.
Dans Colab : **Fichier → Importer le notebook**, puis exécuter la cellule d'installation
dans une nouvelle session avant les autres cellules. Python 3.12 ou ultérieur requis.
Si Colab demande de redémarrer la session après installation, accepter puis reprendre
depuis le début. Enregistrer une copie du notebook et télécharger ses sorties avant de quitter.

Le CPU suffit. Ces estimateurs scikit-learn ne deviennent pas des modèles GPU en sélectionnant un accélérateur.
Internet est nécessaire pour installer les bibliothèques; les données sont synthétiques.
Les ressources Colab ne sont pas garanties : [FAQ officielle](https://research.google.com/colaboratory/faq.html).
Les versions ci-dessous sont celles du cours, pas une recommandation de toujours installer les dernières.


In [ ]:
# Exécuter en premier, dans une session neuve.
import os, sys, subprocess
from importlib.metadata import version as package_version
if sys.version_info < (3, 12):
    raise RuntimeError("Choisir un runtime Python >= 3.12, ou utiliser le pack local Python 3.12.")
required = {'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2'}
if os.environ.get("CYBERSUP_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *[f"{k}=={v}" for k, v in required.items()]])
for key, expected in required.items():
    assert package_version(key) == expected, f"Version incorrecte : {key}. Reprendre dans une session neuve."
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[key] = "2"
print("Python", sys.version.split()[0], "· dépendances", {k: package_version(k) for k in required})


# TP 06 · Expliquer et quantifier l'incertitude
90 minutes. Régression Friedman synthétique. Séparer entraînement, diagnostic, calibration conforme et test.
Livrable : importance par permutation, PDP/ICE, couverture observée et largeur d'intervalle.
Sources : scikit-learn inspection ; Angelopoulos & Bates, arXiv:2107.07511.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import root_mean_squared_error
X,y=make_friedman1(n_samples=1600,n_features=10,noise=2,random_state=42)
Xbase,Xtest,ybase,ytest=train_test_split(X,y,test_size=.2,random_state=42)
Xtr,Xrest,ytr,yrest=train_test_split(Xbase,ybase,test_size=.375,random_state=43)
Xdiag,Xcal,ydiag,ycal=train_test_split(Xrest,yrest,test_size=.5,random_state=44)
model=HistGradientBoostingRegressor(max_iter=140,max_leaf_nodes=15,l2_regularization=1,random_state=42).fit(Xtr,ytr)
imp=permutation_importance(model,Xdiag,ydiag,n_repeats=5,scoring="neg_root_mean_squared_error",random_state=42)
print(pd.DataFrame({"variable":np.arange(10),"hausse RMSE":imp.importances_mean}).sort_values("hausse RMSE",ascending=False))
PartialDependenceDisplay.from_estimator(model,Xdiag,[0,1],kind="both",subsample=35,random_state=42)
plt.show()

Exercice 1 (25 min). Construire des intervalles split-conformal à 90%. Utiliser le rang fini-échantillon ceil((n_cal+1)(1-alpha)) et non un quantile approximatif ordinaire.

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.

Exercice 2 (20 min). Calculer la couverture séparément pour x0<0.5 et x0>=0.5. Quelle garantie manque ?

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.

Exercice 3 (15 min). Une importance élevée prouve-t-elle qu'agir sur cette variable améliore la cible ? Donner un contre-exemple.

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.